<a href="https://colab.research.google.com/github/sabriddinovna/graph_nn/blob/main/simple_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model=resnet18(weights=ResNet18_Weights.DEFAULT)
data=torch.rand(1,3,64,64)
labels=torch.rand(1,1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]


In [2]:
prediction=model(data)

In [4]:
loss=(prediction-labels).sum()
loss.backward()

In [5]:
optim=torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

In [6]:
optim.step()

In [7]:
import torch
a=torch.tensor([2. ,3. ], requires_grad=True)
b=torch.tensor([2. ,3. ], requires_grad=True)

In [10]:
Q=3*a**3-b**2

In [11]:
external_grad= torch.tensor([1. , 1. ])
Q.backward(gradient=external_grad)

In [12]:
from torch import nn, optim
model=resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
  param.requires_grad=False

In [13]:
model.fc=nn.Linear(512, 10)

In [14]:
opt=optim.SGD(model.parameters(),  lr=1e-2,  momentum=0.9)

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [16]:
class Net(nn.Module):
  def __init__(self):
    super().__init__()

    # 1 input img channel, 6 output channels, 5x5 square conv
    self.conv1=nn.Conv2d(1, 6, 5)
    self.conv2=nn.Conv2d(6, 16, 5)
    self.fc1=nn.Linear(16*5*5, 120) # 5x5 img dimension
    self.fc2=nn.Linear(120,84)
    self.fc3=nn.Linear(84, 10)

  def forward(self,input):
    c1=F.relu(self.conv1(input))
    s2=F.max_pool2d(c1, (2,2))
    c3=F.relu(self.conv2(s2))
    s4=F.max_pool2d(c3, 2)
    s4=torch.flatten(s4, 1)
    f5=F.relu(self.fc1(s4))
    f6=F.relu(self.fc2(f5))

    output=self.fc3(f6)
    return output

In [17]:
net= Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [18]:
params=list(net.parameters())
print(len(params))

10


In [19]:
print(params[0].size())

torch.Size([6, 1, 5, 5])


In [20]:
input=torch.randn(1,1, 32,32)
out=net(input)
print(out)

tensor([[-0.0147, -0.1859, -0.1598,  0.0264,  0.0965,  0.0503,  0.0192, -0.0664,
          0.0353,  0.0004]], grad_fn=<AddmmBackward0>)


In [21]:
output=net(input)

In [22]:
target=torch.randn(10)
target=target.view(1,-1)
criterion=nn.MSELoss()
loss=criterion(output, target)

In [23]:
print(loss)

tensor(1.8455, grad_fn=<MseLossBackward0>)


input -> conv2d -> relu -> maxpool2d -> conv2d -> relu -> maxpool2d
      -> flatten -> linear -> relu -> linear -> relu -> linear
      -> MSELoss
      -> loss

In [24]:
print(loss.grad_fn)  # MSELoss
print(loss.grad_fn.next_functions[0][0])  # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])  # ReLU

In [25]:
net.zero_grad()
print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)
loss.backward()
print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
None
conv1.bias.grad after backward
tensor([-0.0156,  0.0124,  0.0224, -0.0323,  0.0214, -0.0079])


weight=weight-learning_rate*gradient

In [27]:
lr=0.01
for f in net.parameters():
  with torch.no_grad():
    f=f-f.grad*lr

In [29]:
import torch.optim as optim
optimizer=optim.SGD(net.parameters(), lr=0.01)

optimizer.zero_grad()
output=net(input)
loss=criterion(output, target)
loss.backward()
optimizer.step()